# Map Coordinates 

## 1. Data Files

The data files to be loaded are:

- Attraction file name: attractions_with_coordinates.csv  


## Data Description

Assume the data files already exist locally.

The datasets have the following structure:

### attractions_with_coordinates.csv 
- Each row represents one attraction.
- Columns:
  - [totalScore] (float): overall rating of the attraction.
  - [latitude] (float): latitude of the attraction.
  - [longitude] (float): longitude of the attraction.



## 2. What I Want to See on the Map

I want the map to show:

- All attraction locations  

For each attraction marker, I want to display:

- The attraction number and its overall rating.



## 3. Final Output

The final output should be:

- An interactive map showing all attractions.


## Response Rules for GenAI

Please strictly follow these rules:

1. Assume all CSV files already exist locally in the working directory.
2. The first action in the code must be loading the three CSV files using pandas.
3. Do NOT check whether the files exist and do NOT ask me to upload them.
4. Do NOT comment on execution environment limitations.
5. Only generate code needed to load the data and display the map.
6. Do NOT perform any optimization, modeling, or analysis.
7. Do NOT introduce additional datasets or advanced features.
8. Use `pandas` and `folium`.
9. Keep the code simple and easy to run.
10. Do NOT add long explanations or non-code text.


## Task for GenAI
Please generate **Python code** that:

- Check and install required packages (`pandas`, `folium`) if necessary
- Imports required libraries (`pandas`, `folium`).
- Loads [the CSV file "attractions_with_coordinates.csv"].
- Creates [a map that shows all attraction locations. For each attraction marker, I want to display the attraction number and its overall rating].
- Displays attraction locations with simple popups.
- Shows the map as the final output.

**Only output the code.**

In [1]:
import sys
import subprocess
import importlib

for package in ["pandas", "folium"]:
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import folium

attractions = pd.read_csv("attractions_with_coordinates_price.csv")

center_lat = attractions["latitude"].mean()
center_lon = attractions["longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

for i, row in attractions.reset_index(drop=True).iterrows():
    popup_text = f"Attraction {i+1}<br>Name: {row['title']}<br>Rating: {row['totalScore']}"
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=folium.Popup(popup_text, max_width=250)
    ).add_to(m)

m

# All possible constraints without weights in code

Now I want to make a function with arbitrary constraints as we do not know what are the actual requests of every user here. So I wrote all the possible combinations of constraints the users may choose. Please let me know what am I missing or optimise my constraints. 

Suppose that I have n attractions, m categories and I plan to visit n_selected attractions. Here are my criterias:

Variables:
- y_i: Binary variable where attraction i is selected (Decision Variable)
- r_i: Review score of attraction i 
- e_i: Entrance fee of each attractions, column name "prices" 
- A : A set of attraction which is a subset of {1,...,n}
- C : A set of categories which is a subset of {1,...,m}
- z_j: A binary variable that indicate if category j is represented at least once 
- a_{ij}: Binary variable if attraction i is truly category j 


Define category count for category j: 
- N_j = sum{i = 1 to n} a_{ij}*y_i

Group of categories C:
- N_c = sum{i = 1 to n}1[sum{j is an element of C}(a_{ij} >= 1)]* y_i

Objective function: sum{i = 1 to n}(r_i * y_i)

s.t. 
1. sum{i = 1 to n} y_i = n_selected (must sure the n_selected >= 1)

2. If I want to visit attraction a, I must visit attraction b
- y_a <= y_b

3. If all attractions in precondition set P are visited, then all attractions in postcondition set Q must be visited
- sum{i is an element of P} y_i - |P| + 1 <= y_k for all k in Q

4. If any attraction in P is visited, then at least one in Q must be visited 
- sum{i is an element of P} y_i <= |P| sum{k is an element of Q} y_k

5. If I want to visit at least one category a, I must visit at least one category b
- sum{i = 1 to n} a_{ia} * y_i <= n * sum{i = 1 to n} a_{ib}*y_i

6. If all categories in set C_1 are present, then all categories in set C_2 is present 
- sum{j is an element of C_1} z_j - |C_1| + 1 <= z_k for all k in C_2 where C_1 and C_2 are subsets of C

7. I will visit at most one attraction total from attraction sets G_1 and G_2
- sum{i is an element of G_1} y_i + sum{k is element of G_2} y_k <= 1 where G_1 and G_2 are subsets of A

8. I will not select attractions from both groups simultaneously, but may select many within one group 
- sum{i is an element of G_1} y_i <= |G_1|u where u: A binary variable if G_1 is chosen, u = 1, otherwise 0 
- sum{i is an element of G_2} y_i <= |G_2|(1-u) where u: A binary variable if G_1 is chosen, u = 1, otherwise 0 

9. I will not visit both sets of categories C_1 and C_2 simultaneously, but I can select any categories within one group
- sum{i is an element of C_1} z_i <= |C_1|u where u: A binary variable if C_1 is chosen, u = 1, otherwise 0 
- sum{i is an element of C_2} z_i <= |C_2|(1-u) where u: A binary variable if C_1 is chosen, u = 1, otherwise 0 

10. I cannot visit both attractions a and b 
- y_a + y_b <= 1

11. I cannot visit both categories a and b 
- z_a + z_b <= 1

12. If I visit at least p attractions of category a, I must at least visit q attractions of category b
- N_a >= pt
- N_a <= (p-1) + Mt
- N_b >= qt
where t is a binary variable whether the precondition occurs, t is an element of (0,1). 

13. I want to visit at most p attractions of a category set C
- sum{i = 1 to n} g_i * y_i <= p where g_i = 1 if attraction i belongs to at least one category in group C

14. I want to visit at least p attractions of a category set C
- sum{i = 1 to n} g_i * y_i >= p where g_i = 1 if attraction i belongs to at least one category in group C

15. I want to visit at most p attractions of category a  
- sum{i = 1 to n} a_{ia} * y_i <= p 

16. I want to visit at least p attractions of category a 
- sum{i = 1 to n} a_{ia} * y_i >= p 

17. Lower and upper bounds of p attractions are selected in category C
- L<= sum{i = 1 to n} g_i * y_i <= U where g_i = 1 if attraction i belongs to at least one category in group C, L is the lower bound and U is the upper bound

18. I want to visit at most p attractions from city j
- sum{i is an element of a set that contains "city j" under the "city" column in attractions} y_i <= p

19. I want to visit at least p attractions from city j
- sum{i is an element of a set that contains "city j" under the "city" column in attractions} y_i >= p

20. I want to visit exactly p attractions from city j 
- sum{i is an element of a set that contains "city j" under the "city" column in attractions} y_i = p

21. I do not want to visit attractions i 
- y_i = 0 for all i in A_{forbidden}

22. I do not want to visit category j
- sum{i=1 to n} a_{ij}*y_i = 0

23. I want to visit a group of attractions i 
- y_i = 1 for all i in A_{want to visit}

23. I want to visit one attraction from category j 
- sum{i = 1 to n} a_{ia} * y_i = 1

24. y_i and z_j is an element of {0,1}

25. sum{i = 1 to n} e_i * y_i <= E, where E is my budget and by default, set it as the average price * number of attractions I intended to go (i.e. 9 attractions here)

Once I have defined all the possible constraints any users may think of, I want to create function called constraints_without_weight_crafting() where I just input the constraints the user wants. Then, I can later apply this function to the solve_attraction_finding_problem().

In [2]:
def _prepare_attraction_dataframe(df: pd.DataFrame, weighted=False):

    df = df.copy().reset_index(drop=True)

    # -------------------------
    # Title
    # -------------------------
    if "title" not in df.columns:
        if "name" in df.columns:
            df["title"] = df["name"]
        else:
            df["title"] = [f"Attraction_{i+1}" for i in range(len(df))]

    # -------------------------
    # Score column detection
    # -------------------------
    if "review_score" in df.columns:
        score_col = "review_score"
    elif "totalScore" in df.columns:
        score_col = "totalScore"
    elif "rating" in df.columns:
        score_col = "rating"
    else:
        raise ValueError("No review score column found.")

    # -------------------------
    # Prices
    # -------------------------
    if "prices" not in df.columns:
        df["prices"] = 0

    df[score_col] = pd.to_numeric(df[score_col], errors="coerce").fillna(0)
    df["prices"] = pd.to_numeric(df["prices"], errors="coerce").fillna(0)

    # -------------------------
    # Categories → all_categories
    # -------------------------
    category_cols = [col for col in df.columns if "categories" in col.lower()]

    if len(category_cols) == 0:
        raise ValueError("No category columns found.")

    df["all_categories"] = df[category_cols].values.tolist()
    df["all_categories"] = df["all_categories"].apply(
        lambda x: [str(i).strip() for i in x if pd.notna(i) and str(i).strip() != ""]
    )

    # -------------------------
    # Normalization (only if weighted)
    # -------------------------
    if weighted:
        r_min, r_max = df[score_col].min(), df[score_col].max()
        e_min, e_max = df["prices"].min(), df["prices"].max()

        df["r_i_norm"] = 1.0 if r_max == r_min else (df[score_col] - r_min) / (r_max - r_min)
        df["e_i_norm"] = 0.0 if e_max == e_min else (df["prices"] - e_min) / (e_max - e_min)

    return df, score_col

In [3]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB, quicksum


def constraints_without_weight_crafting(
    model: gp.Model,
    attractions: pd.DataFrame,
    y,
    n_select: int,
    constraints: list | None = None,
    budget: float | None = None,
):
    """
    Add arbitrary user-defined constraints to a Gurobi attraction-selection model
    without weighted preference crafting.

    Parameters
    ----------
    model : gp.Model
        Existing Gurobi model.
    attractions : pd.DataFrame
        Attraction dataframe. Expected columns:
        - 'prices'
        - 'city'
        - one or more columns containing 'categories'
        - optional 'title' or 'name'
    y : gurobipy.tupledict
        Binary selection variables y[i].
    n_select : int
        Exact number of attractions to select.
    constraints : list[dict] or None
        User-defined constraints. Supported types are listed below.
    budget : float or None
        Total budget. If None, default = average price * n_select.

    Returns
    -------
    artifacts : dict
        Dictionary containing helper objects:
        - model
        - y
        - z
        - one_hot_encoded
        - category_to_idx
        - attraction_to_idx
        - city_to_idx
        - added_constraints
    """

    df = attractions.copy().reset_index(drop=True)

    # -------------------------------------------------
    # Basic columns
    # -------------------------------------------------
    if "title" not in df.columns:
        if "name" in df.columns:
            df["title"] = df["name"]
        else:
            df["title"] = [f"Attraction_{i+1}" for i in range(len(df))]

    if "city" not in df.columns:
        df["city"] = ""

    if "prices" not in df.columns:
        df["prices"] = 0

    df["prices"] = pd.to_numeric(df["prices"], errors="coerce").fillna(0)

    # -------------------------------------------------
    # Build one-hot category matrix
    # -------------------------------------------------
    category_cols = [col for col in df.columns if "categories" in col.lower()]
    if not category_cols:
        raise ValueError("No category columns found. Please ensure category columns contain 'categories'.")

    # df["all_categories"] = df[category_cols].values.tolist()
    # df["all_categories"] = df["all_categories"].apply(
    #     lambda x: [str(v).strip() for v in x if pd.notna(v) and str(v).strip() != ""]
    # )

    categories_exploded = df[["all_categories"]].explode("all_categories")
    one_hot = pd.get_dummies(categories_exploded["all_categories"])
    one_hot_encoded = one_hot.groupby(categories_exploded.index).sum()
    one_hot_encoded = one_hot_encoded.reindex(df.index, fill_value=0)

    # -------------------------------------------------
    # Sets and mappings
    # -------------------------------------------------
    I = range(len(df))
    category_names = list(one_hot_encoded.columns)
    J = range(len(category_names))

    attraction_to_idx = {str(df.loc[i, "title"]).strip().lower(): i for i in I}
    category_to_idx = {str(category_names[j]).strip().lower(): j for j in J}

    unique_cities = sorted(df["city"].astype(str).str.strip().str.lower().unique())
    city_to_idx = {city: idx for idx, city in enumerate(unique_cities)}

    # -------------------------------------------------
    # a[i, j] parameter
    # -------------------------------------------------
    a = {
        (i, j): int(one_hot_encoded.iloc[i, j])
        for i in I for j in J
    }

    # -------------------------------------------------
    # Budget default
    # -------------------------------------------------
    if budget is None:
        budget = float(df["prices"].mean() * n_select)

    e = df["prices"].to_dict()

    # -------------------------------------------------
    # Base constraints
    # -------------------------------------------------
    added_constraints = []

    c_exact = model.addConstr(quicksum(y[i] for i in I) == n_select, name="exactly_n_selected")
    added_constraints.append(c_exact)

    c_budget = model.addConstr(quicksum(e[i] * y[i] for i in I) <= budget, name="budget_limit")
    added_constraints.append(c_budget)

    # -------------------------------------------------
    # Category presence variables z[j]
    # z[j] = 1 iff category j is represented at least once
    # -------------------------------------------------
    z = model.addVars(J, vtype=GRB.BINARY, name="z")

    model.addConstrs(
        (z[j] <= quicksum(a[i, j] * y[i] for i in I) for j in J),
        name="z_lb"
    )
    model.addConstrs(
        (quicksum(a[i, j] * y[i] for i in I) <= n_select * z[j] for j in J),
        name="z_ub"
    )

    # -------------------------------------------------
    # Helper functions
    # -------------------------------------------------
    def _attraction_idx(name):
        key = str(name).strip().lower()
        if key not in attraction_to_idx:
            raise ValueError(f"Attraction '{name}' not found.")
        return attraction_to_idx[key]

    def _category_idx(name):
        key = str(name).strip().lower()
        if key not in category_to_idx:
            raise ValueError(f"Category '{name}' not found.")
        return category_to_idx[key]

    def _attraction_idx_list(names):
        return list(dict.fromkeys(_attraction_idx(name) for name in names))

    def _category_idx_list(names):
        return list(dict.fromkeys(_category_idx(name) for name in names))

    def _city_indicator(city_name):
        city_name = str(city_name).strip().lower()
        return {
            i: int(city_name in str(df.loc[i, "city"]).strip().lower())
            for i in I
        }

    def _group_indicator_from_categories(category_names_group):
        cat_ids = _category_idx_list(category_names_group)
        return {
            i: int(sum(a[i, j] for j in cat_ids) >= 1)
            for i in I
        }

    # -------------------------------------------------
    # Supported arbitrary constraints
    # -------------------------------------------------
    # Each item in constraints is a dictionary with field "type"
    # -------------------------------------------------
    if constraints is None:
        constraints = []

    trigger_count = 0
    chooser_count = 0

    for idx, cons in enumerate(constraints, start=1):
        ctype = cons["type"]

        # 2. If attraction a, then attraction b
        if ctype == "if_attraction_then_attraction":
            a_idx = _attraction_idx(cons["a"])
            b_idx = _attraction_idx(cons["b"])
            c = model.addConstr(y[a_idx] <= y[b_idx], name=f"if_attr_then_attr_{idx}")
            added_constraints.append(c)

        # 3. If all attractions in P, then all in Q
        elif ctype == "if_all_attractions_then_all_attractions":
            P = _attraction_idx_list(cons["P"])
            Q = _attraction_idx_list(cons["Q"])
            for q in Q:
                c = model.addConstr(
                    quicksum(y[i] for i in P) - len(P) + 1 <= y[q],
                    name=f"if_allP_then_q_{idx}_{q}"
                )
                added_constraints.append(c)

        # 4. If any attraction in P, then at least one in Q
        elif ctype == "if_any_attraction_then_any_attraction":
            P = _attraction_idx_list(cons["P"])
            Q = _attraction_idx_list(cons["Q"])
            c = model.addConstr(
                quicksum(y[i] for i in P) <= len(P) * quicksum(y[k] for k in Q),
                name=f"if_anyP_then_anyQ_{idx}"
            )
            added_constraints.append(c)

        # 5. If category a, then category b
        elif ctype == "if_category_then_category":
            a_idx = _category_idx(cons["a"])
            b_idx = _category_idx(cons["b"])
            c = model.addConstr(z[a_idx] <= z[b_idx], name=f"if_cat_then_cat_{idx}")
            added_constraints.append(c)

        # 6. If all categories in C1, then all categories in C2
        elif ctype == "if_all_categories_then_all_categories":
            C1 = _category_idx_list(cons["C1"])
            C2 = _category_idx_list(cons["C2"])
            for k in C2:
                c = model.addConstr(
                    quicksum(z[j] for j in C1) - len(C1) + 1 <= z[k],
                    name=f"if_allC1_then_ck_{idx}_{k}"
                )
                added_constraints.append(c)

        # 7. At most one attraction total from G1 union G2
        elif ctype == "at_most_one_total_from_two_attraction_groups":
            G1 = _attraction_idx_list(cons["G1"])
            G2 = _attraction_idx_list(cons["G2"])
            total_group = list(dict.fromkeys(G1 + G2))
            c = model.addConstr(
                quicksum(y[i] for i in total_group) <= 1,
                name=f"at_most_one_from_two_groups_{idx}"
            )
            added_constraints.append(c)

        # 8. Cannot choose both attraction groups simultaneously
        elif ctype == "not_both_attraction_groups":
            G1 = _attraction_idx_list(cons["G1"])
            G2 = _attraction_idx_list(cons["G2"])
            chooser_count += 1
            u = model.addVar(vtype=GRB.BINARY, name=f"u_attr_group_{chooser_count}")
            c1 = model.addConstr(
                quicksum(y[i] for i in G1) <= len(G1) * u,
                name=f"attr_group1_choice_{idx}"
            )
            c2 = model.addConstr(
                quicksum(y[i] for i in G2) <= len(G2) * (1 - u),
                name=f"attr_group2_choice_{idx}"
            )
            added_constraints.extend([c1, c2])

        # 9. Cannot choose both category groups simultaneously
        elif ctype == "not_both_category_groups":
            C1 = _category_idx_list(cons["C1"])
            C2 = _category_idx_list(cons["C2"])
            chooser_count += 1
            u = model.addVar(vtype=GRB.BINARY, name=f"u_cat_group_{chooser_count}")
            c1 = model.addConstr(
                quicksum(z[j] for j in C1) <= len(C1) * u,
                name=f"cat_group1_choice_{idx}"
            )
            c2 = model.addConstr(
                quicksum(z[j] for j in C2) <= len(C2) * (1 - u),
                name=f"cat_group2_choice_{idx}"
            )
            added_constraints.extend([c1, c2])

        # 10. Not both attractions a and b
        elif ctype == "not_both_attractions":
            a_idx = _attraction_idx(cons["a"])
            b_idx = _attraction_idx(cons["b"])
            c = model.addConstr(y[a_idx] + y[b_idx] <= 1, name=f"not_both_attr_{idx}")
            added_constraints.append(c)

        # 11. Not both categories a and b
        elif ctype == "not_both_categories":
            a_idx = _category_idx(cons["a"])
            b_idx = _category_idx(cons["b"])
            c = model.addConstr(z[a_idx] + z[b_idx] <= 1, name=f"not_both_cat_{idx}")
            added_constraints.append(c)

        # 12. If at least p of category a, then at least q of category b
        elif ctype == "if_category_count_at_least_p_then_category_count_at_least_q":
            a_idx = _category_idx(cons["a"])
            b_idx = _category_idx(cons["b"])
            p = int(cons["p"])
            q = int(cons["q"])
            M = n_select
            trigger_count += 1
            t = model.addVar(vtype=GRB.BINARY, name=f"t_cat_count_{trigger_count}")

            N_a = quicksum(a[i, a_idx] * y[i] for i in I)
            N_b = quicksum(a[i, b_idx] * y[i] for i in I)

            c1 = model.addConstr(N_a >= p * t, name=f"cat_count_trigger_lb_{idx}")
            c2 = model.addConstr(N_a <= (p - 1) + M * t, name=f"cat_count_trigger_ub_{idx}")
            c3 = model.addConstr(N_b >= q * t, name=f"cat_count_consequence_{idx}")
            added_constraints.extend([c1, c2, c3])

        # 13. Category group max
        elif ctype == "category_group_max":
            cats = cons["categories"]
            p = int(cons["value"])
            g = _group_indicator_from_categories(cats)
            c = model.addConstr(
                quicksum(g[i] * y[i] for i in I) <= p,
                name=f"cat_group_max_{idx}"
            )
            added_constraints.append(c)

        # 14. Category group min
        elif ctype == "category_group_min":
            cats = cons["categories"]
            p = int(cons["value"])
            g = _group_indicator_from_categories(cats)
            c = model.addConstr(
                quicksum(g[i] * y[i] for i in I) >= p,
                name=f"cat_group_min_{idx}"
            )
            added_constraints.append(c)

        # 15. Category max
        elif ctype == "category_max":
            j = _category_idx(cons["category"])
            p = int(cons["value"])
            c = model.addConstr(
                quicksum(a[i, j] * y[i] for i in I) <= p,
                name=f"cat_max_{idx}"
            )
            added_constraints.append(c)

        # 16. Category min
        elif ctype == "category_min":
            j = _category_idx(cons["category"])
            p = int(cons["value"])
            c = model.addConstr(
                quicksum(a[i, j] * y[i] for i in I) >= p,
                name=f"cat_min_{idx}"
            )
            added_constraints.append(c)

        # 17. Category group bounds
        elif ctype == "category_group_bounds":
            cats = cons["categories"]
            L = int(cons["L"])
            U = int(cons["U"])
            g = _group_indicator_from_categories(cats)
            c = model.addConstr(
                quicksum(g[i] * y[i] for i in I) >= L,
                name=f"cat_group_lb_{idx}"
            )
            c2 = model.addConstr(
                quicksum(g[i] * y[i] for i in I) <= U,
                name=f"cat_group_ub_{idx}"
            )
            added_constraints.extend([c, c2])

        # 18. City max
        elif ctype == "city_max":
            h = _city_indicator(cons["city"])
            p = int(cons["value"])
            c = model.addConstr(
                quicksum(h[i] * y[i] for i in I) <= p,
                name=f"city_max_{idx}"
            )
            added_constraints.append(c)

        # 19. City min
        elif ctype == "city_min":
            h = _city_indicator(cons["city"])
            p = int(cons["value"])
            c = model.addConstr(
                quicksum(h[i] * y[i] for i in I) >= p,
                name=f"city_min_{idx}"
            )
            added_constraints.append(c)

        # 20. City exact
        elif ctype == "city_exact":
            h = _city_indicator(cons["city"])
            p = int(cons["value"])
            c = model.addConstr(
                quicksum(h[i] * y[i] for i in I) == p,
                name=f"city_exact_{idx}"
            )
            added_constraints.append(c)

        # 21. Forbidden attractions
        elif ctype == "attraction_forbidden":
            A_forbid = _attraction_idx_list(cons["attractions"])
            for i in A_forbid:
                c = model.addConstr(y[i] == 0, name=f"forbid_attr_{idx}_{i}")
                added_constraints.append(c)

        # 22. Forbidden category
        elif ctype == "category_forbidden":
            j = _category_idx(cons["category"])
            c = model.addConstr(
                quicksum(a[i, j] * y[i] for i in I) == 0,
                name=f"forbid_cat_{idx}"
            )
            added_constraints.append(c)

        # 23. Required attractions
        elif ctype == "attraction_required":
            A_req = _attraction_idx_list(cons["attractions"])
            for i in A_req:
                c = model.addConstr(y[i] == 1, name=f"require_attr_{idx}_{i}")
                added_constraints.append(c)

        # 24. Exactly one attraction from category
        elif ctype == "category_exact":
            j = _category_idx(cons["category"])
            c = model.addConstr(
                quicksum(a[i, j] * y[i] for i in I) == 1,
                name=f"cat_exact_{idx}"
            )
            added_constraints.append(c)

        # Attraction set max
        elif ctype == "attraction_set_max":
            A_set = _attraction_idx_list(cons["attractions"])
            p = int(cons["value"])
            c = model.addConstr(quicksum(y[i] for i in A_set) <= p, name=f"attr_set_max_{idx}")
            added_constraints.append(c)

        # Attraction set min
        elif ctype == "attraction_set_min":
            A_set = _attraction_idx_list(cons["attractions"])
            p = int(cons["value"])
            c = model.addConstr(quicksum(y[i] for i in A_set) >= p, name=f"attr_set_min_{idx}")
            added_constraints.append(c)

        # Attraction set exact
        elif ctype == "attraction_set_exact":
            A_set = _attraction_idx_list(cons["attractions"])
            p = int(cons["value"])
            c = model.addConstr(quicksum(y[i] for i in A_set) == p, name=f"attr_set_exact_{idx}")
            added_constraints.append(c)

        else:
            raise ValueError(f"Unsupported constraint type: {ctype}")

    return {
        "model": model,
        "y": y,
        "z": z,
        "one_hot_encoded": one_hot_encoded,
        "category_to_idx": category_to_idx,
        "attraction_to_idx": attraction_to_idx,
        "city_to_idx": city_to_idx,
        "added_constraints": added_constraints,
    }

#### Input the constraints into solve_attraction_searcher_problem

In [4]:
def solve_attraction_searcher_problem(attractions, n_select=9, constraints=None, budget=None):

    df, score_col = _prepare_attraction_dataframe(attractions, weighted=False)

    model = gp.Model("attraction_finding")
    model.setParam("OutputFlag", 0)

    I = range(len(df))
    y = model.addVars(I, vtype=GRB.BINARY, name="y")

    r = df[score_col].to_dict()

    model.setObjective(quicksum(r[i] * y[i] for i in I), GRB.MAXIMIZE)

    artifacts = constraints_without_weight_crafting(
        model=model,
        attractions=df,
        y=y,
        n_select=n_select,
        constraints=constraints,
        budget=budget,
    )

    model.optimize()

    if model.status == GRB.OPTIMAL:
        selected_idx = [i for i in I if y[i].X > 0.5]
        selected_df = df.loc[selected_idx].copy()

        cols = [c for c in ["title", score_col, "prices", "city", "latitude", "longitude", "all_categories"] if c in selected_df.columns]
        selected_df = selected_df[cols].reset_index(drop=True)

        total_budget_required = selected_df["prices"].sum()

        print(f"Objective value: {model.objVal:.4f}")

        # ✅ Print user budget (if provided)
        if budget is not None:
            print(f"User budget (yen): {budget:.0f}")
        else:
            print("User budget: Not specified (model determined optimal selection)")

        # ✅ Print required budget
        print(f"Budget required (yen): {total_budget_required:.0f}")

        # ✅ Optional: show remaining budget
        if budget is not None:
            remaining = budget - total_budget_required
            print(f"Remaining budget (yen): {remaining:.0f}")

        print(selected_df.to_string(index=False))

        return selected_df, model, artifacts

    return None, model, artifacts

In [5]:
CONSTRAINT_TYPES = [

    # Core
    "exact_n",
    "budget",

    # Attraction-level
    "attraction_required",
    "attraction_forbidden",
    "if_attraction_then_attraction",
    "if_all_attractions_then_all_attractions",
    "if_any_attraction_then_any_attraction",
    "not_both_attractions",
    "at_most_one_total_from_two_attraction_groups",
    "not_both_attraction_groups",
    "attraction_set_min",
    "attraction_set_max",
    "attraction_set_exact",

    # Category-level
    "category_min",
    "category_max",
    "category_exact",
    "category_forbidden",
    "if_category_then_category",
    "not_both_categories",
    "if_category_count_at_least_p_then_category_count_at_least_q",

    # Category-group
    "category_group_min",
    "category_group_max",
    "category_group_bounds",
    "if_all_categories_then_all_categories",
    "not_both_category_groups",

    # City-level
    "city_min",
    "city_max",
    "city_exact"
]

In [6]:
"attraction_required" * 2 
"if_any_attraction_then_any_attraction"
"category_max" * 3
"if_category_then_category"
"city_min"
"city_exact"

'city_exact'

In [32]:
# Example constraints input

constraints = [
    {"type": "attraction_required", "attractions":["Imperial Palace East National Gardens", "Shibuya Sky"]},
    {"type": "attraction_forbidden", "attractions":["Tokyo Camii & Diyanet Turkish Culture Center"]},
    {"type": "if_any_attraction_then_any_attraction", "P":["Sensō-ji", "Cat Street"], "Q":["Tokyo Dome", "Ueno Park"]},
    {"type": "category_max", "category": "Museum", "value": 2},
    {"type": "category_min", "category": "Buddhist temple", "value": 1},
    {"type": "category_exact", "category": "Garden"},
    {"type": "category_forbidden", "category": "Tourist information center"},
    {"type": "city_max", "city": "Shinjuku", "value": 3},
    {"type": "category_group_min", "categories": ["Amusement park", "Amusement center", "Theme park"], "value": 1},
    {"type": "category_min", "category": "Bar", "value": 1},
    {"type": "if_attraction_then_attraction", "a": "Sensō-ji", "b": "Tokyo Skytree"},
    {"type": "city_min", "city": "Taito", "value": 1}
]

In [8]:
constraints2 = [
    {"type": "if_all_attractions_then_all_attractions", "P": ["Samurai Ninja Museum Asakusa Tokyo", "Yoyogi Park"], "Q": ["Ninja Trick House In Tokyo", "Tokyo National Museum"]},
    {"type": "not_both_attractions", "a": "Unicorn Gundam", "b": "Ninja Trick House In Tokyo"},
    {"type": "at_most_one_total_from_two_attraction_groups", "G1": ["Samurai Ninja Museum Asakusa Tokyo", "Yoyogi Park"], "G2": ["Tokyo Camii & Diyanet Turkish Culture Center", "Cat Street"]},
    {"type": "attraction_set_max", "attractions": ["Tokyo Tower", "Tokyo National Museum", "Meiji Jingu"], "value": 2},
    {"type": "attraction_set_min", "attractions": ["Imperial Palace East National Gardens", "Shinjuku Gyoen National Garden", "Yoyogi Park"], "value": 1},
    {"type": "attraction_set_exact", "attractions": ["Sensō-ji", "Tokyo Skytree", "Asakusa Hanayashiki"], "value": 2},
    {"type": "if_category_then_category", "a": "Buddhist temple", "b":"garden"},
    {"type": "not_both_categories", "a": "Amusement park", "b": "Theme park"},
    {"type": "category_group_max", "categories": ["Shopping mall", "Market", "Bar"], "value": 2},
    {"type": "category_group_bounds", "categories": ["Buddhist temple", "Bar", "Observation Deck", "Garden"], "L": 1, "U":3},
    {"type": "not_both_category_groups", "C1": ["Mosque", "Historical landmark"], "C2": ["Buddhist temple", "Art gallery"]},
    {"type": "city_exact", "city": "Sumida", "value": 2}
]

In [9]:
constraints3 = [
    {"type": "not_both_attraction_groups", "G1": ["Samurai Ninja Museum Asakusa Tokyo", "Yoyogi Park"], "G2": ["Ninja Trick House In Tokyo", "Tokyo National Museum"]},    
    {"type": "if_category_count_at_least_p_then_category_count_at_least_q", "a": "Observation Deck", "b":"garden", "p": 0, "q": 2},
    {"type": "if_all_categories_then_all_categories", "C1": ["Mosque", "Historical landmark"], "C2": ["Buddhist temple", "Art gallery"]},
    {"type": "category_exact", "category": "Historical Landmark", "value": 1}
]

In [10]:
# constraints = [
#     {"type": "attraction_forbidden", "attractions": ["Tokyo Camii & Diyanet Turkish Culture Center"]},
#     {"type": "if_attraction_then_attraction", "a": "Tokyo Dome City", "b": "Shinjuku Gyoen Cherry Tree Area"},
#     {"type": "attraction_set_max", "attractions": ["Ninja Trick House In Tokyo", "Samurai Ninja Museum Asakusa Tokyo", "Asakusa Hanayashiki"], "value": 2},
#     {"type": "category_max", "category": "Shopping mall", "value": "2"},
#     {"type": "category_forbidden", "category": "Tourist information center"},
#     {"type": "not_both_categories", "a": "Buddhist temple", "b": "Shinto shrine"},
#     {"type": "category_group_min", "categories": ["Amusement park", "Amusement center", "Theme park"], "value": 2},
#     {"type": "city_max", "city": "Shinjuku", "value": 3}
# ]

In [11]:
solve_attraction_searcher_problem(
    attractions=attractions,
    n_select=6,
    constraints=constraints,
    budget=10000
)

Restricted license - for non-production use only - expires 2027-11-29


(None,
 <gurobi.Model MIP instance attraction_finding: 137 constrs, 125 vars, Parameter changes: OutputFlag=0>,
 {'model': <gurobi.Model MIP instance attraction_finding: 137 constrs, 125 vars, Parameter changes: OutputFlag=0>,
  'y': {0: <gurobi.Var y[0]>,
   1: <gurobi.Var y[1]>,
   2: <gurobi.Var y[2]>,
   3: <gurobi.Var y[3]>,
   4: <gurobi.Var y[4]>,
   5: <gurobi.Var y[5]>,
   6: <gurobi.Var y[6]>,
   7: <gurobi.Var y[7]>,
   8: <gurobi.Var y[8]>,
   9: <gurobi.Var y[9]>,
   10: <gurobi.Var y[10]>,
   11: <gurobi.Var y[11]>,
   12: <gurobi.Var y[12]>,
   13: <gurobi.Var y[13]>,
   14: <gurobi.Var y[14]>,
   15: <gurobi.Var y[15]>,
   16: <gurobi.Var y[16]>,
   17: <gurobi.Var y[17]>,
   18: <gurobi.Var y[18]>,
   19: <gurobi.Var y[19]>,
   20: <gurobi.Var y[20]>,
   21: <gurobi.Var y[21]>,
   22: <gurobi.Var y[22]>,
   23: <gurobi.Var y[23]>,
   24: <gurobi.Var y[24]>,
   25: <gurobi.Var y[25]>,
   26: <gurobi.Var y[26]>,
   27: <gurobi.Var y[27]>,
   28: <gurobi.Var y[28]>,
   29

In [12]:
selected_df, model, artifacts = solve_attraction_searcher_problem(
    attractions=attractions,
    n_select=7,
    constraints=constraints,
    budget=10000
)

Objective value: 30.9000
User budget (yen): 10000
Budget required (yen): 6500
Remaining budget (yen): 3500
                                title  totalScore  prices    city  latitude  longitude                                                        all_categories
                        Tokyo Skytree         4.4    3100  Sumida 35.710054 139.810714                                [Observation deck, Tourist attraction]
                             Sensō-ji         4.5       0   Taito 35.713403 139.795526                                 [Buddhist temple, Tourist attraction]
Imperial Palace East National Gardens         4.4       0 Chiyoda 35.683847 139.750686   [National park, Garden, National reserve, Park, Tourist attraction]
                          Shibuya Sky         4.6    3400 Shibuya 35.658286 139.702262                                [Observation deck, Tourist attraction]
                      Tokyo Dome City         4.3       0  Bunkyo 35.705571 139.751970  [Amusement center, B

### What I Already Have

I already have:

- Optimal attractions and their latitude and longitude in selected_df["title", "latitude", "longitude"]
- A folium map named [m]

## Response Rules for GenAI

Please strictly follow these rules:

1. Do NOT reimport any libraries that have been imported before.
2. Output **Python code only**.

# Task for GenAI

- Provide code to plot the attractions on the folium map [m_optimised]
- I am only interested in all the attractions in selected_df 

In [13]:
m_optimised = folium.Map(
location=[selected_df["latitude"].mean(), selected_df["longitude"].mean()],
zoom_start=12
)

for _, row in selected_df[["title", "latitude", "longitude"]].dropna(subset=["latitude", "longitude"]).iterrows():
    folium.Marker(
    location=[row["latitude"], row["longitude"]],
    popup=row["title"],
    tooltip=row["title"]
    ).add_to(m_optimised)

m_optimised

# All possible constraints with weights in code

Now I want to try the same case as the weights. Please let me know what am I missing or optimise my constraints. 

Suppose that I have n attractions, m categories and I plan to visit n_selected attractions. Here are my criterias:

Here is my reference setting:

| Preference           | Review Weight (w_r) | Entrance Fee Weight (w_e) |
|---------------------|--------------------|---------------------------|
| Premium Experience  | 0.8                | 0.2                       |
| Balanced Experience | 0.5                | 0.5                       |
| Budget-friendly     | 0.2                | 0.8                       |

Variables:
- y_i: Binary variable where attraction i is selected (Decision Variable)
- r_i: Review score of attraction i 
- e_i: Entrance fee of each attractions, column name "prices" 
- A : A set of attraction which is a subset of {1,...,n}
- C : A set of categories which is a subset of {1,...,m}
- z_j: A binary variable that indicate if category j is represented at least once 


Define category count for category j: 
- N_j = sum{i = 1 to n} a_{ij}*y_i

Group of categories C:
- N_c = sum{i = 1 to n}1[sum{j is an element of C}(a_{ij} >= 1)]* y_i

Objective function: max sum{i = 1 to n}((w_r* r_i^norm - w_e* e_i^norm)* y_i)

s.t. 
1. sum{i = 1 to n} y_i = n_selected (must sure the n_selected >= 1)

2. If I visit attraction a, I must visit attraction b
- y_a <= y_b

3. If all attractions in precondition set P are visited, then all attractions in postcondition set Q must be visited
- sum{i is an element of P} y_i - |P| + 1 <= y_k for all k in Q

4. If any attraction in P is visited, then at least one in Q must be visited 
- sum{i is an element of P} y_i <= |P| sum{k is an element of Q} y_k

5. If I visit at least one category a, I must visit at least one category b
- sum{i = 1 to n} a_{ia} * y_i <= n * sum{i = 1 to n} a_{ib}*y_i

6. If all categories in set C_1 are present, then all categories in set C_2 is present 
- sum{j is an element of C_1} z_j - |C_1| + 1 <= z_k for all k in C_2 where C_1 and C_2 are subsets of C

7. I will visit at most one attraction total from attraction sets G_1 and G_2
- sum{i is an element of G_1} y_i + sum{i is element of G_2} y_i <= 1 where G_1 and G_2 are subsets of A

8. I will not select attractions from both groups simultaneously, but may select many within one group 
- sum{i is an element of G_1} y_i <= |G_1|u where u: A binary variable if G_1 is chosen, u = 1, otherwise 0 
- sum{i is an element of G_2} y_i <= |G_2|(1-u) where u: A binary variable if G_1 is chosen, u = 1, otherwise 0 

9. I will not visit both sets of categories C_1 and C_2 simultaneously, but I can select any categories within one group
- sum{j is an element of C_1} z_j <= |C_1|u where u: A binary variable if C_1 is chosen, u = 1, otherwise 0 
- sum{j is an element of C_2} z_j <= |C_2|(1-u) where u: A binary variable if C_1 is chosen, u = 1, otherwise 0 

10. I cannot visit both attractions a and b 
- y_a + y_b <= 1

11. I cannot visit both categories a and b 
- z_a + z_b <= 1

12. If I visit at least p attractions of category a, I must at least visit q attractions category b
- N_a >= pt
- N_a <= (p-1) + Mt
- N_b >= qt
where t is a binary variable whether the precondition occurs, t is an element of (0,1). 

13. I visit at most p attractions of a category set C
- sum{i = 1 to n} g_i * y_i <= p where g_i = 1 if attraction i belongs to at least one category in group C

14. I visit at least p attractions of a category set C
- sum{i = 1 to n} g_i * y_i >= p where g_i = 1 if attraction i belongs to at least one category in group C

15. I visit at most p attractions of category a  
- sum{i = 1 to n} a_{ia} * y_i <= p 

16. I visit at least p attractions of category a 
- sum{i = 1 to n} a_{ia} * y_i >= p 

17. Lower and upper bounds of p attractions are selected category C
- L<= sum{i = 1 to n} g_i * y_i <= U where g_i = 1 if attraction i belongs to at least one category in group C, L is the lower bound and U is the upper bound

18. I want to visit at most p attractions from city j
- sum{i is an element of a set that contains "city j" under the "city" column in attractions} y_i <= p

19. I want to visit at least p attractions from city j
- sum{i is an element of a set that contains "city j" under the "city" column in attractions} y_i >= p

20. I want to visit p attractions from city j 
- sum{i is an element of a set that contains "city j" under the "city" column in attractions} y_i = p

21. I do not want to visit attractions i 
- y_i = 0 for all i in A_{forbidden}

22. I do not want to visit category j
- sum{i=1 to n} a_{ij}*y_i = 0

23. I will want to visit attractions i 
- y_i = 1 for all i in A_{want to visit}

23. I will want to visit one attraction from category j 
- sum{i = 1 to n} a_{ia} * y_i = 1

24. y_i and z_j is an element of {0,1}

25. sum{i = 1 to n} e_i * y_i <= E, where E is my budget and by default, set it as the average price * number of attractions I intended to go (i.e. 9 attractions here)

26. (w_r​,w_e​) is an element of {(0.8,0.2),(0.5,0.5),(0.2,0.8)}

I want to see two columns: r_i^norm and e_i^norm in the df
- r_i^norm = (r_i - r_min)/(r_max - r_min)
- e_i^norm = (e_i - e_min)/(e_max - e_min)

Once I have defined all the possible constraints any users may think of, I want to create function called constraints_with_weight_crafting() where I just input the constraints the user wants. Then, I can later apply this function to the solve_attraction_finding_weighted().

In [14]:
def constraints_with_weight_crafting(
    model,
    attractions: pd.DataFrame,
    y,
    n_select,
    constraints=None,
    preference="Premium Experience",
    budget=None
):

    df = attractions.copy().reset_index(drop=True)

    # -------------------------------------------------
    # 1. Normalize scores
    # -------------------------------------------------
    r = pd.to_numeric(df["totalScore"], errors="coerce").fillna(0)
    e = pd.to_numeric(df["prices"], errors="coerce").fillna(0)

    r_min, r_max = r.min(), r.max()
    e_min, e_max = e.min(), e.max()

    if r_max == r_min:
        df["r_norm"] = 1
    else:
        df["r_norm"] = (r - r_min) / (r_max - r_min)

    if e_max == e_min:
        df["e_norm"] = 0
    else:
        df["e_norm"] = (e - e_min) / (e_max - e_min)

    r_norm = df["r_norm"].to_dict()
    e_norm = df["e_norm"].to_dict()

    # -------------------------------------------------
    # 2. Select weights
    # -------------------------------------------------
    weights = {
        "Premium Experience": (0.8, 0.2),
        "Balanced Experience": (0.5, 0.5),
        "Budget-friendly": (0.2, 0.8),
    }

    if preference not in weights:
        raise ValueError("Invalid preference")

    w_r, w_e = weights[preference]

    # -------------------------------------------------
    # 3. Set objective
    # -------------------------------------------------
    I = range(len(df))

    model.setObjective(
        gp.quicksum((w_r * r_norm[i] - w_e * e_norm[i]) * y[i] for i in I),
        GRB.MAXIMIZE
    )

    # -------------------------------------------------
    # 4. Reuse all constraints
    # -------------------------------------------------
    artifacts = constraints_without_weight_crafting(
        model=model,
        attractions=df,
        y=y,
        n_select=n_select,
        constraints=constraints,
        budget=budget
    )

    # -------------------------------------------------
    # 5. Return everything
    # -------------------------------------------------
    artifacts["r_norm"] = df["r_norm"]
    artifacts["e_norm"] = df["e_norm"]
    artifacts["weights"] = (w_r, w_e)

    return artifacts

In [15]:
def solve_attraction_finding_weighted(
    attractions,
    n_select=9,
    constraints=None,
    budget=None,
    preference="Premium Experience"
):

    df, score_col = _prepare_attraction_dataframe(attractions, weighted=True)

    weights = {
        "Premium Experience": (0.8, 0.2),
        "Balanced Experience": (0.5, 0.5),
        "Budget-friendly": (0.2, 0.8),
    }

    if preference not in weights:
        raise ValueError("Invalid preference")

    w_r, w_e = weights[preference]

    model = gp.Model("attraction_finding_weighted")
    model.setParam("OutputFlag", 0)

    I = range(len(df))
    y = model.addVars(I, vtype=GRB.BINARY, name="y")

    r_norm = df["r_i_norm"].to_dict()
    e_norm = df["e_i_norm"].to_dict()

    model.setObjective(
        quicksum((w_r * r_norm[i] - w_e * e_norm[i]) * y[i] for i in I),
        GRB.MAXIMIZE
    )

    artifacts = constraints_without_weight_crafting(
        model=model,
        attractions=df,
        y=y,
        n_select=n_select,
        constraints=constraints,
        budget=budget,
    )

    model.optimize()

    if model.status == GRB.OPTIMAL:
        selected_idx = [i for i in I if y[i].X > 0.5]
        selected_df = df.loc[selected_idx].copy()

        cols = [c for c in [
            "title", score_col, "prices",
            "r_i_norm", "e_i_norm",
            "city", "latitude", "longitude", "all_categories"
        ] if c in selected_df.columns]

        selected_df = selected_df[cols].reset_index(drop=True)

        selected_df["weighted_score"] = (
            w_r * selected_df["r_i_norm"] - w_e * selected_df["e_i_norm"]
        )

        total_budget_required = selected_df["prices"].sum()

        print(f"Preference: {preference}")
        print(f"Weights -> review: {w_r}, entrance fee: {w_e}")
        print(f"Objective value: {model.objVal:.4f}")
        print(f"Budget required (yen): {total_budget_required:.0f}")
        print(selected_df.to_string(index=False))

        return selected_df, model, artifacts

    return None, model, artifacts

In [16]:
selected_df_weighted, model, artifacts = solve_attraction_finding_weighted(
    attractions=attractions,
    n_select=15,
    constraints=constraints,
    budget=None,
    preference="Budget-friendly"
)

Preference: Budget-friendly
Weights -> review: 0.2, entrance fee: 0.8
Objective value: 1.0167
Budget required (yen): 6500
                                   title  totalScore  prices  r_i_norm  e_i_norm    city  latitude  longitude                                                        all_categories  weighted_score
                           Tokyo Skytree         4.4    3100  0.642857  0.645833  Sumida 35.710054 139.810714                                [Observation deck, Tourist attraction]       -0.388095
                                Sensō-ji         4.5       0  0.714286  0.000000   Taito 35.713403 139.795526                                 [Buddhist temple, Tourist attraction]        0.142857
   Imperial Palace East National Gardens         4.4       0  0.642857  0.000000 Chiyoda 35.683847 139.750686   [National park, Garden, National reserve, Park, Tourist attraction]        0.128571
                             Shibuya Sky         4.6    3400  0.785714  0.708333 Shibuya 35.65

In [17]:
m_optimised = folium.Map(
location=[selected_df_weighted["latitude"].mean(), selected_df["longitude"].mean()],
zoom_start=12
)

for _, row in selected_df_weighted[["title", "latitude", "longitude"]].dropna(subset=["latitude", "longitude"]).iterrows():
    folium.Marker(
    location=[row["latitude"], row["longitude"]],
    popup=row["title"],
    tooltip=row["title"]
    ).add_to(m_optimised)

m_optimised

In [18]:
selected_df.rename(columns={"totalScore": "review_score"}, inplace=True)
selected_df_weighted.rename(columns={"totalScore": "review_score"}, inplace=True)

In [19]:
selected_df.to_csv("selected_attractions_without_weights_sample.csv")
selected_df_weighted.to_csv("selected_attractions_with_weights_sample.csv")

## Solving for feasibility of n_selected values for both weighted and non-weighted models

#### `check_feasibility_by_n_select(attractions, n_range, constraints=None, budget=None)`

This function tests whether your attraction selection model remains feasible when varying the number of attractions (n_select), given a set of constraints and an optional budget.

Inputs:
- attractions: DataFrame containing attraction data
- n_range: List or range of values for number of attractions to select
- constraints (optional): Custom user-defined constraints
- budget (optional): Budget constraint

Output:
- Returns a DataFrame summarising feasibility for each n_select

In [20]:
def check_feasibility_by_n_select(attractions, n_range, constraints=None, budget=None):
    results = []

    for n_select in n_range:
        df, score_col = _prepare_attraction_dataframe(attractions, weighted=False)

        model = gp.Model(f"feasibility_check_{n_select}")
        model.setParam("OutputFlag", 0)

        I = range(len(df))
        y = model.addVars(I, vtype=GRB.BINARY, name="y")

        # Dummy objective: feasibility only
        model.setObjective(0, GRB.MAXIMIZE)

        artifacts = constraints_without_weight_crafting(
            model=model,
            attractions=df,
            y=y,
            n_select=n_select,
            constraints=constraints,
            budget=budget,
        )

        model.optimize()

        if model.status == GRB.OPTIMAL:
            selected_idx = [i for i in I if y[i].X > 0.5]
            results.append({
                "n_select": n_select,
                "feasible": True,
                "selected_count": len(selected_idx)
            })
        else:
            results.append({
                "n_select": n_select,
                "feasible": False,
                "selected_count": None
            })

    return pd.DataFrame(results)

In [21]:
feasibility_table = check_feasibility_by_n_select(
    attractions=attractions,
    n_range=range(1, 16),
    constraints=constraints,
    budget=7000
)

print(feasibility_table)

    n_select  feasible  selected_count
0          1     False             NaN
1          2     False             NaN
2          3     False             NaN
3          4     False             NaN
4          5     False             NaN
5          6     False             NaN
6          7      True             7.0
7          8      True             8.0
8          9      True             9.0
9         10      True            10.0
10        11      True            11.0
11        12      True            12.0
12        13      True            13.0
13        14      True            14.0
14        15      True            15.0


#### `diagnose_infeasibility(attractions, n_select, constraints=None, budget=None)`

This function checks whether the attraction selection model is feasible for a given number of selected attractions (n_select).
If the model is infeasible, it uses Gurobi’s IIS (Irreducible Inconsistent Subsystem) feature to identify which constraints are causing the conflict.

Inputs
- attractions: DataFrame containing the attraction data
- n_select: Exact number of attractions to be selected
- constraints (optional): User-defined constraints
- budget (optional): Budget limit

Output
- Returns the Gurobi model object
- Prints whether the model is feasible or infeasible
- If infeasible, prints the constraint names involved in the IIS

In [22]:
def diagnose_infeasibility(attractions, n_select, constraints=None, budget=None):
    df, score_col = _prepare_attraction_dataframe(attractions, weighted=False)

    model = gp.Model(f"infeasibility_diagnosis_{n_select}")
    model.setParam("OutputFlag", 0)

    I = range(len(df))
    y = model.addVars(I, vtype=GRB.BINARY, name="y")

    model.setObjective(0, GRB.MAXIMIZE)

    artifacts = constraints_without_weight_crafting(
        model=model,
        attractions=df,
        y=y,
        n_select=n_select,
        constraints=constraints,
        budget=budget,
    )

    model.optimize()

    if model.status == GRB.INFEASIBLE:
        print(f"Model is infeasible for n_select = {n_select}")
        model.computeIIS()

        print("\nConstraints in IIS:")
        for c in model.getConstrs():
            if c.IISConstr:
                print(c.ConstrName)

    elif model.status == GRB.OPTIMAL:
        print(f"Model is feasible for n_select = {n_select}")
    else:
        print(f"Model ended with status {model.status}")

    return model

In [33]:
diagnose_infeasibility(
    attractions=attractions,
    n_select=5,
    constraints=constraints,
    budget=7000
)

Model is infeasible for n_select = 5

Constraints in IIS:
exactly_n_selected
require_attr_1_3
if_anyP_then_anyQ_3
cat_min_5
cat_group_min_9
cat_min_10
if_attr_then_attr_11


<gurobi.Model MIP instance infeasibility_diagnosis_5: 137 constrs, 125 vars, Parameter changes: OutputFlag=0>

In [ ]:
## Put the link(s) to your chatlog here:
https://chatgpt.com/share/69dfbc47-babc-839f-9ff2-e85b1be45519